# 🛰️ SAM + Land Cover Classification
**Orthophoto segmentation with Segment Anything Model**

Pipeline:
1. Load orthophoto from Google Drive (windowed reading)
2. SAM automatic segmentation on tiles
3. K-means clustering for land cover classification
4. Export classified GeoTIFF

> ⚠️ Runtime → Change runtime type → **T4 GPU**

In [6]:
from google.colab import drive
drive.mount('/content/drive')

SyntaxError: invalid syntax (1671707684.py, line 1)

In [1]:
!pip install -q segment-anything rasterio shapely scikit-learn matplotlib tqdm
!pip install -q opencv-python-headless

# Download SAM ViT-B checkpoint (~375MB, fits T4 comfortably)
import os
if not os.path.exists('sam_vit_b_01ec64.pth'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
    print('✅ SAM ViT-B checkpoint downloaded')
else:
    print('✅ Checkpoint already exists')

✅ SAM ViT-B checkpoint downloaded


In [3]:
import numpy as np
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import cv2
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from sklearn.cluster import MiniBatchKMeans
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Auto-find .tif files in your Drive
import glob
tif_files = glob.glob('/content/drive/MyDrive/**/*.tif', recursive=True)
tif_files += glob.glob('/content/drive/MyDrive/**/*.tiff', recursive=True)

if tif_files:
    print(f'Found {len(tif_files)} GeoTIFF file(s):')
    for i, f in enumerate(tif_files):
        size_mb = os.path.getsize(f) / 1e6
        print(f'  [{i}] {f}  ({size_mb:.1f} MB)')
    # Auto-select first one, or change index
    ORTHO_PATH = tif_files[0]
    print(f'\n✅ Using: {ORTHO_PATH}')
    print('👆 Wrong file? Change: ORTHO_PATH = tif_files[N]')
else:
    print('❌ No .tif files found in Drive. Check your upload.')
    ORTHO_PATH = '/content/drive/MyDrive/your_orthophoto.tif'  # fallback

assert os.path.exists(ORTHO_PATH), f'❌ File not found: {ORTHO_PATH}'

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Inspect orthophoto metadata (no full load)
with rasterio.open(ORTHO_PATH) as src:
    print(f'Size: {src.width} x {src.height}')
    print(f'Bands: {src.count}')
    print(f'CRS: {src.crs}')
    print(f'Pixel size: {src.res}')
    print(f'Bounds: {src.bounds}')
    print(f'Dtype: {src.dtypes[0]}')
    meta = src.meta.copy()
    bounds = src.bounds
    transform = src.transform
    crs = src.crs
    W, H = src.width, src.height

print(f'\nTotal pixels: {W*H:,.0f} ({W*H/1e9:.1f}B)')
print(f'Estimated uncompressed: {W*H*4/1e9:.1f} GB')

In [ ]:
# Preview: read a small region (2048x2048) from center
cx, cy_px = W // 2, H // 2
preview_size = 2048
window = Window(cx - preview_size//2, cy_px - preview_size//2, preview_size, preview_size)

with rasterio.open(ORTHO_PATH) as src:
    # Read RGB (bands 1-3), skip alpha
    preview = src.read([1, 2, 3], window=window)
    preview = np.moveaxis(preview, 0, -1)  # CHW → HWC

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.imshow(preview)
ax.set_title(f'Preview: center {preview_size}x{preview_size} px ({preview_size*0.0162:.0f}m x {preview_size*0.0162:.0f}m)')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Load SAM model
sam = sam_model_registry['vit_b'](checkpoint='sam_vit_b_01ec64.pth')
sam.to(device)
sam.eval()

mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,        # density of point grid
    pred_iou_thresh=0.86,      # quality threshold
    stability_score_thresh=0.92,
    min_mask_region_area=100,  # filter tiny segments (in pixels)
)
print('✅ SAM loaded on', device)

In [ ]:
# Tiling configuration
TILE_SIZE = 1024       # SAM works best at 1024x1024
OVERLAP = 64           # overlap to reduce edge artifacts
STRIDE = TILE_SIZE - OVERLAP

# ========================================
# 👇 处理区域：先跑一小块试试
# ========================================
# Option A: Process a small demo region (fast, ~2 min)
DEMO_MODE = True
DEMO_TILES_X = 5   # 5x5 tiles = ~5k x 5k px
DEMO_TILES_Y = 5

# Starting position (center of image)
START_X = W // 2 - (DEMO_TILES_X * STRIDE) // 2
START_Y = H // 2 - (DEMO_TILES_Y * STRIDE) // 2

if DEMO_MODE:
    n_tiles_x, n_tiles_y = DEMO_TILES_X, DEMO_TILES_Y
    region_w = DEMO_TILES_X * STRIDE + OVERLAP
    region_h = DEMO_TILES_Y * STRIDE + OVERLAP
    print(f'Demo mode: {n_tiles_x}x{n_tiles_y} tiles = {region_w}x{region_h} px')
    print(f'Coverage: {region_w*0.0162:.0f}m x {region_h*0.0162:.0f}m')
else:
    START_X, START_Y = 0, 0
    n_tiles_x = (W - OVERLAP) // STRIDE + 1
    n_tiles_y = (H - OVERLAP) // STRIDE + 1
    print(f'Full image: {n_tiles_x}x{n_tiles_y} = {n_tiles_x*n_tiles_y} tiles')
    print(f'⚠️ This will take a LONG time on free Colab')

print(f'Total tiles: {n_tiles_x * n_tiles_y}')

In [ ]:
# Run SAM on tiles, collect all segment features
all_features = []   # (mean_r, mean_g, mean_b, std_r, std_g, std_b, area)
all_masks = []      # list of (mask, tile_offset_x, tile_offset_y)

with rasterio.open(ORTHO_PATH) as src:
    total = n_tiles_x * n_tiles_y
    pbar = tqdm(total=total, desc='SAM segmentation')

    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):
            x0 = START_X + tx * STRIDE
            y0 = START_Y + ty * STRIDE

            # Clamp to image bounds
            x0 = min(x0, W - TILE_SIZE)
            y0 = min(y0, H - TILE_SIZE)

            window = Window(x0, y0, TILE_SIZE, TILE_SIZE)
            tile = src.read([1, 2, 3], window=window)
            tile = np.moveaxis(tile, 0, -1)  # HWC

            # Skip mostly-nodata tiles (alpha/black)
            if tile.mean() < 5:
                pbar.update(1)
                continue

            # Run SAM
            masks = mask_generator.generate(tile)

            for m in masks:
                seg = m['segmentation']  # bool mask
                area = seg.sum()
                if area < 50:
                    continue

                pixels = tile[seg]  # (N, 3)
                feat = [
                    pixels[:, 0].mean(), pixels[:, 1].mean(), pixels[:, 2].mean(),
                    pixels[:, 0].std(),  pixels[:, 1].std(),  pixels[:, 2].std(),
                    area
                ]
                all_features.append(feat)
                all_masks.append((seg, x0, y0))

            pbar.update(1)
    pbar.close()

print(f'\n✅ Total segments: {len(all_features)}')

In [ ]:
# K-means land cover classification
N_CLASSES = 6  # 👈 adjust: vegetation, bare soil, water, buildings, roads, shadow

features = np.array(all_features)
# Normalize features
feat_norm = features.copy()
for i in range(features.shape[1]):
    col = features[:, i]
    mn, mx = col.min(), col.max()
    if mx > mn:
        feat_norm[:, i] = (col - mn) / (mx - mn)

kmeans = MiniBatchKMeans(n_clusters=N_CLASSES, random_state=42, batch_size=1000)
labels = kmeans.fit_predict(feat_norm)

# Show cluster centers (mean RGB)
print('Cluster centers (mean R, G, B):')
for i in range(N_CLASSES):
    cluster_mask = labels == i
    center = features[cluster_mask, :3].mean(axis=0)
    count = cluster_mask.sum()
    print(f'  Class {i}: RGB=({center[0]:.0f}, {center[1]:.0f}, {center[2]:.0f}), segments={count}')

In [ ]:
# Render classified result for the demo region
if DEMO_MODE:
    region_w = DEMO_TILES_X * STRIDE + OVERLAP
    region_h = DEMO_TILES_Y * STRIDE + OVERLAP
else:
    region_w, region_h = W, H

# Create classification raster (use uint8)
class_map = np.full((min(region_h, H), min(region_w, W)), 255, dtype=np.uint8)

for idx, (seg, x0, y0) in enumerate(tqdm(all_masks, desc='Rendering')):
    lbl = labels[idx]
    # Map tile coords to region coords
    rx = x0 - START_X
    ry = y0 - START_Y
    # Place segment in class_map
    rows, cols = np.where(seg)
    map_rows = ry + rows
    map_cols = rx + cols
    # Bounds check
    valid = (map_rows >= 0) & (map_rows < class_map.shape[0]) & \
            (map_cols >= 0) & (map_cols < class_map.shape[1])
    class_map[map_rows[valid], map_cols[valid]] = lbl

# Color map
colors = ['#2d6a4f', '#95d5b2', '#dda15e', '#bc6c25', '#457b9d', '#264653']
class_names = ['Vegetation (dense)', 'Vegetation (light)', 'Bare soil', 'Built-up', 'Water/Shadow', 'Dark/Other']
cmap = ListedColormap(colors[:N_CLASSES])

# Read RGB for comparison
with rasterio.open(ORTHO_PATH) as src:
    win = Window(START_X, START_Y, min(region_w, W), min(region_h, H))
    rgb = src.read([1, 2, 3], window=win)
    rgb = np.moveaxis(rgb, 0, -1)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))
axes[0].imshow(rgb)
axes[0].set_title('Original RGB')
axes[0].axis('off')

im = axes[1].imshow(class_map, cmap=cmap, vmin=0, vmax=N_CLASSES-1, interpolation='nearest')
axes[1].set_title(f'Land Cover Classification ({N_CLASSES} classes)')
axes[1].axis('off')

# Legend
patches = [plt.Rectangle((0,0),1,1, fc=colors[i]) for i in range(N_CLASSES)]
axes[1].legend(patches, class_names[:N_CLASSES], loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Export classified region as GeoTIFF
from rasterio.transform import from_bounds

output_path = ORTHO_PATH.replace('.tif', '_classified.tif')

with rasterio.open(ORTHO_PATH) as src:
    # Calculate transform for the processed region
    win = Window(START_X, START_Y, class_map.shape[1], class_map.shape[0])
    win_transform = src.window_transform(win)

    out_meta = {
        'driver': 'GTiff',
        'height': class_map.shape[0],
        'width': class_map.shape[1],
        'count': 1,
        'dtype': 'uint8',
        'crs': src.crs,
        'transform': win_transform,
        'compress': 'lzw',
        'nodata': 255
    }

with rasterio.open(output_path, 'w', **out_meta) as dst:
    dst.write(class_map, 1)

print(f'✅ Exported: {output_path}')
print(f'   Size: {os.path.getsize(output_path)/1e6:.1f} MB')

## 🔧 Next Steps

1. **调整类别数** — 修改 `N_CLASSES` 看哪个效果最好
2. **跑更大区域** — 设 `DEMO_MODE = False`（注意时间）
3. **升级模型** — 换 `vit_h` checkpoint 获得更精细的分割
4. **监督分类** — 手动标注几个 segment，训练 RandomForest 替代 k-means
5. **加 NDVI** — 如果 Band 4 是近红外，可以算植被指数辅助分类